# Embedding the VaR backtest report in a notebook

`report.html` is script-driven, so it won't render if you paste it into a
Markdown cell. Run the code cell below instead: `show_report()` embeds the
report inline using an `<iframe srcdoc=...>`, which renders the self-contained
HTML+JS with no file server and no `file://` access &mdash; so it works in
**VS Code notebooks**, classic Notebook, and JupyterLab alike.

`show_report()`:
1. searches the working directory (and its subfolders) for `report.html` and
   `var_backtest_results.csv`;
2. if the CSV is missing &mdash; it's gitignored, so absent in a fresh clone
   &mdash; regenerates it by running `generate_data.py` then `var_backtest.py`
   (needs `pandas`, `numpy`, `scipy`, `openpyxl` in the kernel's environment);
3. bakes the CSV into the page, shows it inline, and also saves a standalone
   `report_embedded.html` you can open directly in a browser.

**`report.html` vs `report_embedded.html`**
- `report.html` is the reusable report app from the repo: it has a file-picker
  and no data inside, so on its own you'd open it and choose a CSV.
- `report_embedded.html` is what this cell writes each run: a copy of
  `report.html` with *your* CSV baked in, so it shows your data immediately. It's
  disposable (regenerated every run, gitignored) and is what the iframe shows.

**Tips**
- Wrong file auto-picked? Pass explicit paths (forward slashes are fine on
  Windows): `show_report(html_path="C:/path/report.html", csv_path="C:/path/var_backtest_results.csv")`
- Taller/shorter frame: `show_report(height=1200)`.
- If the inline frame is ever blank, open the standalone `report_embedded.html`
  (linked just below the cell output) directly in a browser.

In [ ]:
# Display the VaR report inline. Works in VS Code notebooks, classic Notebook,
# and JupyterLab: the report is embedded via <iframe srcdoc=...>, which renders
# self-contained HTML+JS with no file server or file:// access.
import base64, sys, subprocess
import html as _html
from pathlib import Path
from IPython.display import HTML, display

def _find(name, root, explicit=None):
    if explicit:
        p = Path(explicit).expanduser()
        if not p.exists():
            raise FileNotFoundError(f"{p} not found.")
        return p
    root = Path(root)
    if (root / name).exists():
        return root / name
    hits = sorted(root.rglob(name))   # search subfolders (e.g. a cloned repo dir)
    return hits[0] if hits else None

def show_report(html_path=None, csv_path=None, search_root=None,
                height=1080, generate_if_missing=True):
    root = Path(search_root).expanduser() if search_root else Path.cwd()

    html_file = _find("report.html", root, html_path)
    if html_file is None:
        raise FileNotFoundError(
            f"report.html not found under {root}.\n"
            "Pull the backtest-exceedances repo there, or pass html_path=...")
    repo = html_file.parent

    # The CSV is gitignored, so it may be absent in a fresh clone.
    csv = Path(csv_path) if csv_path else (repo / "var_backtest_results.csv")
    if not csv.exists():
        found = _find("var_backtest_results.csv", root)
        if found is not None:
            csv = found
    if not csv.exists() and generate_if_missing:
        gen, bt = repo / "generate_data.py", repo / "var_backtest.py"
        if gen.exists() and bt.exists():
            print("var_backtest_results.csv missing - generating it from the scripts...")
            try:
                subprocess.run([sys.executable, str(gen)], cwd=repo, check=True)
                subprocess.run([sys.executable, str(bt)], cwd=repo, check=True)
                csv = repo / "var_backtest_results.csv"
            except Exception as e:
                raise RuntimeError(
                    f"Auto-generate failed ({e}). Run generate_data.py then var_backtest.py "
                    f"manually in {repo} (needs pandas, numpy, scipy, openpyxl).")
    if not csv.exists():
        raise FileNotFoundError(
            "var_backtest_results.csv not found (gitignored, so not in a fresh clone).\n"
            f"Run generate_data.py then var_backtest.py in {repo}, or pass csv_path=...")

    # Self-contained page: report.html with this CSV baked in (auto-loads on open).
    page = html_file.read_text(encoding="utf-8")
    b64 = base64.b64encode(csv.read_bytes()).decode()
    page = page.replace(
        "</body>",
        f'<script>window.addEventListener("load",()=>loadText(atob("{b64}")));</script></body>')

    # Save a standalone copy as a fallback you can open directly in a browser.
    out = Path.cwd() / "report_embedded.html"
    out.write_text(page, encoding="utf-8")

    # Inline display: the whole page lives in the srcdoc attribute, so the
    # sandboxed VS Code notebook webview can render it (no server needed).
    srcdoc = _html.escape(page, quote=True)
    iframe = (f'<iframe srcdoc="{srcdoc}" width="100%" height="{height}" '
              'style="border:1px solid #e5e7eb;border-radius:8px"></iframe>')
    p = out.resolve()
    note = ('<div style="font:12px/1.6 -apple-system,Segoe UI,sans-serif;margin-top:6px">'
            'standalone file (open in a browser if the frame is blank): '
            f'<a href="{p.as_uri()}" target="_blank" title="{p}">{p.name}</a></div>')
    display(HTML(iframe + note))

show_report()